# Transit Reliability with LangGraph
###Overview

This project uses LangGraph to orchestrate a modular transit analytics pipeline built on GTFS data. The goal is to compute and rank transit route reliability in a reproducible and structured way. Instead of running a single monolithic script, the computation is broken into independent stages, each responsible for a specific part of the analysis. These stages are connected as a directed graph, where the output of one step becomes the input for the next.

### Architecture

```
GTFS Data Load
      ↓
Basic Metrics (trips, stops)
      ↓
Advanced Metrics (service frequency)
      ↓
Headway Analysis (schedule consistency)
      ↓
Final Scoring & Ranking
      ↓
Output (ranked routes table)
```

In [ ]:
!wget https://marsala-api.cloud.municipiumapp.it/s3/3885/allegati/gtfs/comunedi-marsala-it.zip


--2026-06-19 05:23:26--  https://marsala-api.cloud.municipiumapp.it/s3/3885/allegati/gtfs/comunedi-marsala-it.zip
Resolving marsala-api.cloud.municipiumapp.it (marsala-api.cloud.municipiumapp.it)... 34.128.134.6
Connecting to marsala-api.cloud.municipiumapp.it (marsala-api.cloud.municipiumapp.it)|34.128.134.6|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 137059 (134K) [application/zip]
Saving to: ‘comunedi-marsala-it.zip’

comunedi-marsala-it 100%[===================>] 133.85K   401KB/s    in 0.3s    

2026-06-19 05:23:28 (401 KB/s) - ‘comunedi-marsala-it.zip’ saved [137059/137059]



In [ ]:
!unzip comunedi-marsala-it.zip

Archive:  comunedi-marsala-it.zip
  inflating: agency.txt              
  inflating: areas.txt               
  inflating: calendar.txt            
  inflating: feed_info.txt           
  inflating: routes.txt              
  inflating: shapes.txt              
  inflating: stop_times.txt          
  inflating: stops.txt               
  inflating: trips.txt               


In [ ]:
#Get data
import pandas as pd

def get_data(path):
    data = {
        "routes": pd.read_csv(f"{path}/routes.txt"),
        "trips": pd.read_csv(f"{path}/trips.txt"),
        "stop_times": pd.read_csv(f"{path}/stop_times.txt"),
        "stops": pd.read_csv(f"{path}/stops.txt"),
        "calendar": pd.read_csv(f"{path}/calendar.txt"),
    }
    return data

raw_data = get_data(".")

In [ ]:
for name, df in raw_data.items(): print(name, df.shape)

routes (8, 7)
trips (130, 6)
stop_times (4503, 6)
stops (480, 7)
calendar (2, 10)


In [ ]:
##Basic Metrics
import pandas as pd

def get_basic_metrics(data):
    trips = data["trips"]
    stop_times = data["stop_times"]
    routes = data["routes"]

    # trips per route
    trips_count = trips.groupby("route_id").size().reset_index(name="trip_count")

    # stops per route
    trip_stops = stop_times.groupby("trip_id").size().reset_index(name="stop_count")
    trips_stops = trips.merge(trip_stops, on="trip_id")
    stops_per_route = trips_stops.groupby("route_id")["stop_count"].mean().reset_index(name="avg_stops")

    return trips_count.merge(stops_per_route, on="route_id", how="left")

basic_metrics = get_basic_metrics(raw_data)

In [ ]:
basic_metrics

,route_id,trip_count,avg_stops
0,1905,12,44.5
1,1906,4,46.5
2,1909,30,39.5
3,1912,12,37.0
4,1915,36,14.0
5,1925,12,54.5
6,1944,12,36.0
7,1947,12,47.0


In [ ]:
#Score each route
def calculate_score_routes(df):
    norm = (df - df.min()) / (df.max() - df.min())
    df["score"] = 0.6*norm["trip_count"] + 0.4*norm["avg_stops"]
    return df.sort_values("score", ascending=False)
basic_metrics = calculate_score_routes(basic_metrics)

In [ ]:
basic_metrics

,route_id,trip_count,avg_stops,score
2,1909,30,39.5,0.739352
4,1915,36,14.0,0.600000
5,1925,12,54.5,0.550000
7,1947,12,47.0,0.475926
0,1905,12,44.5,0.451235
3,1912,12,37.0,0.377160
6,1944,12,36.0,0.367284
1,1906,4,46.5,0.320988


In [ ]:
calendar = raw_data['calendar']
trips    = raw_data['trips']

cols = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
calendar[cols] = calendar[cols].astype(int)
days = calendar[cols].sum(axis=1)
days = pd.Series(calendar[cols].sum(axis=1).values, index=calendar["service_id"])
service_days = trips.merge(days.rename("service_days"), left_on="service_id", right_index=True)
service_days.head()

,route_id,service_id,trip_id,trip_headsign,direction_id,shape_id,service_days
0,1925,EST_FER,1925_F_0_1,Piazza Biscione,0,5797,6
1,1925,EST_FER,1925_F_0_2,Piazza Biscione,0,5797,6
2,1925,EST_FER,1925_F_0_3,Piazza Biscione,0,5797,6
3,1925,EST_FER,1925_F_0_4,Piazza Biscione,0,5797,6
4,1925,EST_FER,1925_F_0_5,Piazza Biscione,0,5797,6


In [ ]:


##advanced metrics. calculate route frequecy consistency
## (= trips per service day)
def calculate_advanced_metrics(data, df):
    trips = data["trips"]
    calendar = data["calendar"]

    # active days per service (rough proxy)
    cols = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
    calendar[cols] = calendar[cols].astype(int)
    days = pd.Series(calendar[cols].sum(axis=1).values, index=calendar["service_id"])
    service_days = trips.merge(days.rename("service_days"), left_on="service_id", right_index=True)

    freq = service_days.groupby("route_id")["service_days"].mean().reset_index(name="freq_score")

    return df.merge(freq, on="route_id", how="left")
advanced_metrics = calculate_advanced_metrics(raw_data,basic_metrics)

In [ ]:
advanced_metrics

,route_id,trip_count,avg_stops,score,freq_score
0,1909,30,39.5,0.739352,4.000000
1,1915,36,14.0,0.600000,3.777778
2,1925,12,54.5,0.550000,6.000000
3,1947,12,47.0,0.475926,6.000000
4,1905,12,44.5,0.451235,6.000000
5,1912,12,37.0,0.377160,6.000000
6,1944,12,36.0,0.367284,6.000000
7,1906,4,46.5,0.320988,6.000000


In [ ]:
stop_times = raw_data['stop_times']

In [ ]:
stop_times

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,shape_dist_traveled
0,1925_F_0_1,06:25:00,06:25:00,21000,1,0.0
1,1925_F_0_1,06:25:50,06:25:50,21246,2,330.9
2,1925_F_0_1,06:26:26,06:26:26,20988,3,576.4
3,1925_F_0_1,06:27:01,06:27:01,20993,4,808.8
4,1925_F_0_1,06:27:23,06:27:23,20984,5,956.0
...,...,...,...,...,...,...
4498,1915_D_1_8,20:19:58,20:19:58,20984,10,5987.0
4499,1915_D_1_8,20:20:24,20:20:24,20983,11,6158.2
4500,1915_D_1_8,20:21:03,20:21:03,21001,12,6418.7
4501,1915_D_1_8,20:21:46,20:21:46,21247,13,6707.8


In [ ]:
##How consistent are departture times per route

    # convert HH:MM:SS -> minutes
def to_min(t):
    h, m, s = map(int, t.split(":"))
    return h*60 + m + s/60

def calculate_headway_metrics(data):
    trips = data["trips"]
    stop_times = data["stop_times"]

    # first stop per trip
    stop_times = stop_times.sort_values(["trip_id", "stop_sequence"])
    first_times = stop_times[stop_times["stop_sequence"] == 1][["trip_id","departure_time"]]
    # merge route info
    df = trips[["trip_id", "route_id"]].merge(first_times, on="trip_id")

    df["time_min"] = df["departure_time"].apply(to_min)

    # variability per route (lower = more consistent)
    return df.groupby("route_id")["time_min"].std().reset_index(name="headway_std")

In [ ]:
headway_metrics= calculate_headway_metrics(raw_data)

In [ ]:
def calculate_final_metrics(adv_metrics,hdwy_metrics):
  df1  = advanced_metrics.merge(headway_metrics, on="route_id")
  cols = ["trip_count","avg_stops","freq_score","headway_std"]
  df1[cols] = df1[cols].apply(pd.to_numeric)
  norm = (df1[["trip_count","avg_stops","freq_score","headway_std"]] - df1.min()) / (df1.max() - df1.min())


  df1["final_score"] = (
      0.3*norm["trip_count"] +
      0.2*norm["avg_stops"] +
      0.2*norm["freq_score"] +
      0.3*(1 - norm["headway_std"])
  )

  df1.sort_values("final_score", ascending=False)
  return df1

calculate_final_metrics(advanced_metrics,headway_metrics)

,route_id,trip_count,avg_stops,score,freq_score,headway_std,final_score
0,1909,30,39.5,0.739352,4.000000,272.448666,0.389676
1,1915,36,14.0,0.600000,3.777778,250.958480,0.430677
2,1925,12,54.5,0.550000,6.000000,228.570989,0.741810
3,1947,12,47.0,0.475926,6.000000,223.371209,0.736392
4,1905,12,44.5,0.451235,6.000000,223.112877,0.725617
5,1912,12,37.0,0.377160,6.000000,228.554002,0.655494
6,1944,12,36.0,0.367284,6.000000,230.044462,0.641493
7,1906,4,46.5,0.320988,6.000000,236.550735,0.578781


In [ ]:
##Langraph
from langgraph.graph import StateGraph, END
from typing import TypedDict

class State(TypedDict):
    data: dict
    basic: dict
    advanced: dict
    headway: dict
    final: object

def load_data(state):
    return {"data": raw_data}

def basic(state):
    return {"basic": get_basic_metrics(state["data"]), "data": state["data"]}

def advanced(state):
    return {
        "advanced": calculate_advanced_metrics(state["data"], state["basic"]),
        "data": state["data"]
    }

def headway(state):
    return {
        "headway": calculate_headway_metrics(state["data"]),
        "data": state["data"]
    }

def final(state):
    return {'final': calculate_final_metrics(state['advanced'],state['headway'])
    }


graph = StateGraph(State)

graph.add_node("load", load_data)
graph.add_node("basic", basic)
graph.add_node("advanced", advanced)
graph.add_node("headway", headway)
graph.add_node("final", final)

graph.set_entry_point("load")

graph.add_edge("load", "basic")
graph.add_edge("basic", "advanced")
graph.add_edge("advanced", "headway")
graph.add_edge("headway", "final")

app = graph.compile()

result = app.invoke({})
result["final"]

,route_id,trip_count,avg_stops,score,freq_score,headway_std,final_score
0,1909,30,39.5,0.739352,4.000000,272.448666,0.389676
1,1915,36,14.0,0.600000,3.777778,250.958480,0.430677
2,1925,12,54.5,0.550000,6.000000,228.570989,0.741810
3,1947,12,47.0,0.475926,6.000000,223.371209,0.736392
4,1905,12,44.5,0.451235,6.000000,223.112877,0.725617
5,1912,12,37.0,0.377160,6.000000,228.554002,0.655494
6,1944,12,36.0,0.367284,6.000000,230.044462,0.641493
7,1906,4,46.5,0.320988,6.000000,236.550735,0.578781
